# Topic: SQL CASE WHEN Categorization

## Definition (30-second explanation)
* `CASE WHEN` is SQL's native way of implementing conditional IF/ELSE logic directly within queries.
* It evaluates conditions sequentially from top to bottom, returning the first true match.
* It can be used virtually anywhere in a query: `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, and inside aggregate functions.

## Why Interviewers Ask This
* It is the most heavily used data transformation function in day-to-day analytics.
* Tests your ability to handle business logic, bucketing, and edge cases natively in SQL without relying on Python/Pandas.
* Evaluates your understanding of sequential logic and order of operations.

## Core Concepts
* **Searched CASE:** Uses explicit boolean conditions (`CASE WHEN col > 10 THEN...`).
* **Simple CASE:** Compares a single column to exact values (`CASE col WHEN 'A' THEN...`).
* **Conditional Aggregation:** Combining aggregates with CASE (`SUM(CASE WHEN condition THEN 1 ELSE 0 END)`) to act as a filtered count.
* **Top-to-Bottom Evaluation:** The first matching condition halts evaluation for that row; subsequent true conditions are ignored.

## When to Use
* **Bucketing/Binning:** Converting continuous variables (like salary or age) into categorical tiers.
* **Value Mapping:** Standardizing messy data or creating readable labels (e.g., 'M' to 'Male').
* **Pivoting:** Transforming row values into distinct columns (See 'Pivot' Pattern file).
* **Dynamic Sorting:** Customizing `ORDER BY` logic beyond standard ascending/descending rules.

## Advantages
* Eliminates the need for multiple separate queries by evaluating multiple conditions in a single pass.
* Highly portable standard SQL, unlike dialect-specific shortcuts like `IIF()` or `IF()`.

## Limitations
* Deeply nested `CASE WHEN` statements become extremely difficult to read and maintain.
* Mixing data types in `THEN` branches will cause execution errors.

## Common Comparisons
* **`SUM(CASE...)` vs `COUNT(...)`:** `SUM` with 1/0 is preferred for conditional counting because `COUNT(CASE WHEN condition THEN 1 ELSE 0 END)` will count the 0s as well (since they are non-null). To use `COUNT`, you must omit the `ELSE 0` or use `ELSE NULL`.
* **`CASE` vs `WHERE`:** `WHERE` filters out rows entirely; `CASE` keeps the row but conditionally alters what is displayed or calculated for it.

## Common Interview Traps
* **Missing `ELSE` Clause:** If no condition matches and there is no `ELSE`, it returns `NULL`. Always default to `ELSE 'Unknown'` or `ELSE 0`.
* **Wrong Condition Order:** Putting broad conditions before specific ones (e.g., `> 50` before `> 100` means `> 100` will never trigger).
* **NULL Comparisons:** Writing `CASE WHEN col = NULL` always fails. You must use `CASE WHEN col IS NULL`.
* **Data Type Mismatches:** Returning an integer in one `THEN` and a string in another.

## Python / SQL Syntax
```sql
    -- Pattern 1: Categorization / Bucketing
    SELECT employee_id,
           CASE 
               WHEN salary >= 100000 THEN 'Executive'
               WHEN salary >= 60000 THEN 'Mid-Level'
               ELSE 'Entry Level' 
           END AS salary_tier
    FROM employees;

    -- Pattern 2: Conditional Aggregation
    SELECT department,
           COUNT(*) AS total_employees,
           SUM(CASE WHEN salary >= 80000 THEN 1 ELSE 0 END) AS high_earners
    FROM employees
    GROUP BY department;

    -- Pattern 3: Dynamic Order By
    SELECT * FROM tasks
    ORDER BY 
        CASE WHEN status = 'Urgent' THEN 0 ELSE 1 END,
        due_date ASC;
```

## 45-Second Interview Answer
"I use `CASE WHEN` extensively as it's the most versatile tool for data transformation in SQL. I use it primarily for three patterns: bucketing continuous variables into categories, mapping categorical values, and conditional aggregation—which allows me to calculate multiple filtered metrics in a single pass over the data. I'm always careful to order my conditions from most specific to least specific, ensure data types match across all branches, and include an explicit `ELSE` clause to prevent unexpected NULLs."

## Example Questions:

### Q1: Classify customers into 'New' (< 30 days), 'Regular' (30-365 days), and 'Loyal' (> 365 days) based on registration date.

* **Ideal Interview Answer:** I would use a searched `CASE WHEN` statement to calculate the date difference. I'd order the conditions from smallest to largest so the logic cascades correctly.
```sql
    SELECT customer_id, 
           registration_date,
           CASE 
               WHEN DATEDIFF(CURRENT_DATE, registration_date) < 30 THEN 'New'
               WHEN DATEDIFF(CURRENT_DATE, registration_date) <= 365 THEN 'Regular'
               WHEN registration_date IS NOT NULL THEN 'Loyal'
               ELSE 'Unknown'
           END AS customer_tier
    FROM customers;
```
* **Common Mistakes:** Using overlapping conditions (e.g., `BETWEEN 30 AND 365`) without realizing `CASE WHEN` stops at the first match. For example, explicitly writing `>= 30 AND <= 365` is unnecessary if `< 30` is already handled in the first `WHEN`.
* **Likely Interviewer Follow-up:** How would you handle users who have a NULL registration date? (Answer: My `ELSE 'Unknown'` catches them safely. Alternatively, I could add `WHEN registration_date IS NULL THEN 'Unknown'` as the very first check).

### Q2: Write a query that counts, in a single pass, how many orders are in each status: 'Pending', 'Shipped', 'Delivered', 'Cancelled'.

* **Ideal Interview Answer:** I would use the conditional aggregation pattern using `SUM()` and `CASE WHEN`. This allows me to calculate all four metrics in a single scan of the table without grouping rows.
```sql
    SELECT 
           SUM(CASE WHEN status = 'Pending' THEN 1 ELSE 0 END) AS pending_count,
           SUM(CASE WHEN status = 'Shipped' THEN 1 ELSE 0 END) AS shipped_count,
           SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_count,
           SUM(CASE WHEN status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_count
    FROM orders;
```
* **Common Mistakes:** Trying to use `COUNT(CASE WHEN status = 'Pending' THEN 1 ELSE 0 END)`. `COUNT()` counts all non-null values, so the `0`s from the `ELSE` would still be counted. To use `COUNT`, one must omit the `ELSE 0` so it defaults to NULL, but `SUM` with 1/0 is the standard and safest approach.
* **Likely Interviewer Follow-up:** If we added 20 new statuses tomorrow, what is the drawback of this query? (Answer: It requires hardcoding every status as a new column. If the cardinality is high or dynamic, a standard `GROUP BY status` is much more scalable).

### Q3: For each product, calculate the revenue from weekday orders and weekend orders separately using CASE WHEN with DAYOFWEEK().

* **Ideal Interview Answer:** I'd group by `product_id` and use conditional aggregation. In MySQL, `DAYOFWEEK()` returns 1 for Sunday and 7 for Saturday.
```
    SELECT product_id,
           SUM(CASE WHEN DAYOFWEEK(order_date) IN (2, 3, 4, 5, 6) THEN revenue ELSE 0 END) AS weekday_revenue,
           SUM(CASE WHEN DAYOFWEEK(order_date) IN (1, 7) THEN revenue ELSE 0 END) AS weekend_revenue
    FROM orders
    GROUP BY product_id;
```
* **Common Mistakes:** Forgetting the `ELSE 0` inside the `SUM`. If omitted, it defaults to NULL. While `SUM` usually ignores NULLs, if a product only has weekday sales, its weekend revenue column will evaluate to NULL instead of 0, which can break downstream dashboard visuals.
* **Likely Interviewer Follow-up:** What if your company defines the "weekend" differently depending on the country the order was placed in? (Answer: I would join to a `calendar_dim` or `country_settings` table that has a boolean `is_weekend` flag based on the local timezone/customs, rather than hardcoding numeric logic).

### Q4: Grade employees: 'A' if rating >= 4.5, 'B' if >= 3.5, 'C' if >= 2.5, else 'D'. Count employees per grade per department.

* **Ideal Interview Answer:** I would use a CTE to assign the grade to each employee using the descending threshold logic. Then, in the main query, I would group by department and the new grade.
```sql
    WITH GradedEmployees AS (
        SELECT department,
               CASE 
                   WHEN rating >= 4.5 THEN 'A'
                   WHEN rating >= 3.5 THEN 'B'
                   WHEN rating >= 2.5 THEN 'C'
                   ELSE 'D'
               END AS grade
        FROM employees
    )
    SELECT department, 
           grade, 
           COUNT(*) AS employee_count
    FROM GradedEmployees
    GROUP BY department, grade
    ORDER BY department, grade;
```
* **Common Mistakes:** Putting the conditions out of order (e.g., checking `>= 2.5` first). Because `CASE` evaluates top-to-bottom, everyone above 2.5 would get a 'C', and 'A'/'B' would never trigger.
* **Likely Interviewer Follow-up:** If you didn't use a CTE, how would you write this? (Answer: I would have to copy and paste the entire `CASE WHEN` block into the `GROUP BY` clause, which bloats the code and increases the risk of inconsistencies if I need to update the logic later).

## Practice Questions:

### Q1:
**Scenario:**
You are querying a list of customers to prioritize outreach. The sales team requires a specific sorting order for the output:

- Customers from the 'USA' must always appear at the very top of the list.
- After the USA customers, all other customers should follow, sorted alphabetically by their country.
- Within each country (including the USA), sort the customers by their creditLimit from highest to lowest.

**Mock Schema**
```sql
customerName,country,creditLimit
Signal Gift Stores,USA,71800.00
Mini Gifts,USA,50000.00
Australian Collectors,Australia,117300.00
Atelier graphique,France,21000.00
```

**Your Task:**
Write a MySQL query to return the customerName, country, and creditLimit, sorted exactly according to the sales team's business rules.


* **Answer:** I would use a `CASE WHEN` statement directly inside the `ORDER BY` clause to act as the primary sorting key, assigning 0 to the 'USA' and 1 to everything else. Then, I would add `country ASC` and `creditLimit DESC` as secondary and tertiary sort keys.
```sql
    SELECT customerName, country, creditLimit
    FROM customers
    ORDER BY
        CASE WHEN country = 'USA' THEN 0 ELSE 1 END,
        country ASC,
        creditLimit DESC;
```

* **How Dynamic Sorting Actually Works (The Conceptual Explanation):**
    When you put a `CASE WHEN` inside an `ORDER BY`, the database engine creates a **hidden, temporary column** just for the purpose of sorting. 

    Imagine the database doing this behind the scenes before sorting:
    | customerName | country | creditLimit | *Hidden_Sort_Column* |
    | :--- | :--- | :--- | :--- |
    | Signal Gift Stores | USA | 71800.00 | **0** |
    | Australian Collectors | Australia | 117300.00 | **1** |
    | Atelier graphique | France | 21000.00 | **1** |

    The `ORDER BY` evaluates left-to-right. 
    1. First, it looks at the *Hidden_Sort_Column*. It puts all the `0`s at the top (USA) and all the `1`s below them (everyone else).
    2. Then, for rows tied with a `1`, it looks at the second condition (`country ASC`) and sorts Australia before France.
    3. Finally, if two rows have the exact same country, it looks at the third condition (`creditLimit DESC`) to break the tie.

* **Common Mistakes:** Putting the `CASE WHEN` sorting condition in the `SELECT` clause when the business only asked for it to be sorted, not to output a 0/1 flag.
* **Likely Interviewer Follow-up:** How would you modify this if we wanted 'USA' first, 'Canada' second, and everyone else alphabetical? (Answer: I would expand the CASE statement: `WHEN country = 'USA' THEN 0 WHEN country = 'Canada' THEN 1 ELSE 2 END`).

### Q2:
**Scenario:**
You are interviewing for a Data Analyst/Data Scientist role. The interviewer says: "Our finance team wants to see total revenue grouped by customer, but instead of the years being rows, they need the years to be columns for their dashboard."

**Mock Schema**
```sql
customerNumber,paymentDate,amount
103,2003-06-05,14571.44
103,2004-12-18,1676.14
112,2003-06-06,32641.98
```

* **Answer:** Standard SQL does not have a native `PIVOT` function like Excel or Pandas. To pivot data, I use conditional aggregation. I group by the customer, and for each year column, I use a `SUM` wrapped around a `CASE WHEN` statement. If the payment falls in the target year, it adds the amount; otherwise, it adds 0.
```sql
    SELECT customerNumber,
           SUM(CASE WHEN YEAR(paymentDate) = 2003 THEN amount ELSE 0 END) AS revenue_2003,
           SUM(CASE WHEN YEAR(paymentDate) = 2004 THEN amount ELSE 0 END) AS revenue_2004,
           SUM(CASE WHEN YEAR(paymentDate) = 2005 THEN amount ELSE 0 END) AS revenue_2005
    FROM payments
    GROUP BY customerNumber;
```
* **Common Mistakes:** Forgetting the `GROUP BY` clause, which will cause the query to aggregate the entire table into a single row instead of per customer. Omitting the `ELSE 0`, which will cause `NULL` values in the output for years a customer didn't make a purchase.
* **Likely Interviewer Follow-up:** What happens if the business expands and we now have 20 years of data? Do you have to manually write 20 `SUM(CASE WHEN...)` lines? (Answer: In pure standard SQL, yes, pivoting requires hardcoding the columns. To make it dynamic, you would need to use dynamic SQL via stored procedures to construct the query string programmatically, or handle the pivot in a BI tool or Pandas layer).